## RAG

[RAG 구현하는 방법]
1. 다양한 형식의 문서들을 텍스트로 변환한 뒤 더 작은 chunk로 분할
2. chunk들을 embedding 모델을 이용해 벡터로 변환
3. 벡터 유사도 검색을 통해 질문과 유사한 내용들을 찾음 -> @tool로 구현
4. agent에 tool로 입력

### 1. 다양한 형식의 문서들을 텍스트로 변환하기

실제 데이터들은 굉장히 다양한 형태로 존재합니다. 

word, pdf, excel 등의 파일 내용들을 텍스트로 변환해 봅시다.

#### 1-1. excel 파일 처리하기

엑셀 파일을 처리해 보겠습니다. 엑셀 파일은 아래와 같은 과정으로 데이터를 정리할 수 있습니다.

1. pandas 라이브러리를 이용해 엑셀 파일 로드
2. 엑셀 파일에 존재하는 시트 별로 데이터 처리 시작
3. 각 시트에 존재하는 행 데이터들을 텍스트로 변환하여 저장.
4. 3번 과정을 모든 시트에 대해 수행.
5. 최종적으로 모든 시트의 행 데이터들이 텍스트로 변환되어 종합됨.

In [2]:
import pandas as pd
from pathlib import Path
from langchain_core.documents import Document

def parse_excel(file_path: str) -> list[Document]:
    # 1. 전달받은 경로(file_path)에 있는 엑셀 파일을 읽어옵니다.
    with pd.ExcelFile(file_path) as xl:

        # 2. 데이터를 담아둘 빈 바구니(리스트)를 만듭니다.
        docs = []
        
        # 3. 엑셀 파일 안에 있는 여러 개의 '시트(sheet)'를 하나씩 꺼내서 반복합니다.
        for sheet_name in xl.sheet_names:
            # 4. 현재 시트의 내용을 표(데이터프레임) 형태로 불러옵니다.
            # header=1 은 엑셀의 두 번째 줄(행)을 제목(컬럼 이름)으로 사용한다는 뜻입니다.
            df = xl.parse(sheet_name, header=1)
            
            # 5. 표의 데이터를 한 줄씩 꺼내서 반복합니다.
            for idx, row in df.iterrows():
                # 6. 한 줄에 있는 각 칸의 데이터를 "제목: 내용" 형식으로 묶어서 하나의 긴 글로 만듭니다.
                # (예: "이름: 홍길동 \n 나이: 20")
                text = "\n".join(f"{col}: {row[col]}" for col in df.columns)
                
                # 7. 만들어진 글과 출처 정보를 하나로 묶어서(Document 객체) 바구니에 담습니다.
                docs.append(Document(
                    page_content=text, # 실제 내용이 들어가는 부분
                    metadata={         # 이 데이터가 어디서 왔는지 기록하는 부분
                        "source": file_path, # 어떤 파일에서 왔는지
                        "sheet": sheet_name, # 어떤 시트에서 왔는지
                        "row": idx,          # 몇 번째 줄(행)에서 왔는지
                    }
                ))
            
    # 8. 모든 데이터가 담긴 바구니(문서 리스트)를 최종적으로 반환합니다.
    return docs

excel_contents = parse_excel("data/키키테크_임직원및프로젝트현황.xlsx")
print(f"총 Document 수: {len(excel_contents)}")
print(excel_contents[1].page_content)

총 Document 수: 82
사번: TB-002
성명: 이서연
부서: AI 연구개발본부
팀: LLM팀
직급: 책임연구원
입사일: 2018-07-01
담당업무: RAG 파이프라인 설계 및 최적화
연락처(내선): 5102
이메일: seoyeon.lee@kikitech.co.kr
근무지: 본사
비고: nan


그 뒤에 청크로 나눕니다.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# chunk_size = 한 청크의 최대 토큰 수 / chunk_overlap = 겹치는 영역의 토큰 수
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
excel_chunks = splitter.split_documents(excel_contents)

print(len(excel_chunks))

82


지금 엑셀 파일의 데이터가 한 줄에 그렇게 많지 않기 때문에 분할을 해도 분할이 되지 않습니다!

#### 1-2. PDF 파일 처리

PDF 파일은 PyPDFLoader 라이브러리를 이용해 쉽게 처리할 수 있습니다!

In [4]:
# PDF
from langchain_community.document_loaders import PyPDFLoader

pdf_contents = PyPDFLoader("data/키키테크_AI솔루션_제품카탈로그.pdf").load()

print(len(pdf_contents)) # 12 = PDF 페이지 수
print(pdf_contents[1].page_content) # 각 인덱스의 page_content에는 pdf의 텍스트가 담김.

incorrect startxref pointer(1)
parsing for Object Streams


12
1. 회사 개요 및 솔루션 포트폴리오
1.1 (주)키키테크 소개
(주)키키테크는 2015년 설립된 국내 대표 AI 솔루션 전문기업입니다. 창립 이래 기업의 디지털
전환을 선도하는 혁신적인 AI 기술을 연구·개발해 왔으며, 현재 금융, 제조, 유통, 의료,
법률 등 다양한 산업군의 350여 개 기업 고객을 보유하고 있습니다.
서울 강남구 테헤란로에 본사를 두고 판교, 부산에 지사를 운영 중이며, AI 연구개발 인력
120명을 포함한 총 350명의 임직원이 고객 가치 창출을 위해 노력하고 있습니다. 2024년
매출액 480억원을 기록하며 3년 연속 30% 이상의 성장세를 이어가고 있습니다.
주요 경쟁력
 LLM 파인튜닝 및 RAG 기술 내재화 — 자체 AI 연구팀 보유
 온프레미스/클라우드/하이브리드 배포 모두 지원
 금융위원회 마이데이터 사업자 허가, ISO 27001 인증 보유
 24시간 365일 SLA 기반 기술 지원 체계
 도입 후 6개월 이내 ROI 달성 보장 (조건부)
1.2 솔루션 포트폴리오 전체 구성
 제품명
 카테고리
 주요 대상
 핵심 기술
 TalkBridge Enterprise
 AI 챗봇
 CS, 내부 지식관리
 LLM + RAG
 DataSight AI
 데이터 분석
 경영진, 데이터팀
 NL2SQL + 시각화
 DocuMind
 문서 처리
 법무, 행정, 금융
 OCR + LLM
 PredictCore
 예측 분석
 제조, 물류, 금융
 ML + 이상감지
 AgentFlow
 에이전트 플랫폼
 IT/DevOps, 연구개발
 Multi-Agent


결과를 보면 pdf 문서가 총 12장이므로, 문서도 12장이 생긴걸 볼 수 있습니다.

이제 이 문서의 텍스트를 더 작게 분할해 보겠습니다. 분할에는 `RecursiveCharacterTextSplitter`를 사용할 수 있습니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# chunk_size = 한 청크의 최대 토큰 수 / chunk_overlap = 겹치는 영역의 토큰 수
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
pdf_chunks = splitter.split_documents(pdf_contents)

print(len(pdf_chunks))
print(pdf_chunks[0].page_content)

24


#### 1-3. 워드 파일 처리

워드 파일도 Docx2txtLoader 라는 라이브러리를 사용하면 쉽게 처리할 수 있습니다.

PDF와 마찬가지로 텍스트로 변환한 뒤에 더 작은 텍스트로 분할하면 끝입니다.

In [29]:
# Word
from langchain_community.document_loaders import Docx2txtLoader
word_contents = Docx2txtLoader("data/키키테크_사내규정_행동강령.docx").load()

print(len(word_contents))
print(word_contents[0].page_content)

1
(주)키키테크

임직원 행동강령 및 사내 규정 안내서

Ver. 2025.01 | 인사팀 발행

모든 임직원은 본 안내서를 숙지하고 준수하여야 합니다.

목차

1장. 회사 소개 및 핵심 가치

2장. 임직원 행동강령

3장. 근무 규정

4장. 복리후생 안내

5장. IT 보안 정책

6장. 성희롱·괴롭힘 예방 정책

7장. 비상 연락 및 안전 규정




1장. 회사 소개 및 핵심 가치

1.1 회사 개요

(주)키키테크는 2015년 설립된 AI 솔루션 전문기업으로, 기업용 데이터 분석 플랫폼과 AI 에이전트 개발 서비스를 제공합니다. 현재 서울 강남구 테헤란로 본사를 비롯하여 판교, 부산 지사를 운영하고 있으며, 임직원 수는 약 350명입니다.

회사는 '기술로 연결하는 더 나은 내일'이라는 비전 아래, 고객사의 디지털 전환을 돕는 다양한 AI 솔루션을 연구·개발하고 있습니다.

1.2 핵심 가치 (Core Values)

혁신 (Innovation): 새로운 기술과 아이디어를 두려움 없이 탐구하고 도전합니다.

신뢰 (Trust): 고객, 동료, 파트너와의 관계에서 언제나 투명하고 책임감 있게 행동합니다.

협력 (Collaboration): 다양한 배경과 역량을 가진 구성원들이 함께 더 큰 성과를 만들어냅니다.

성장 (Growth): 개인과 조직이 함께 성장하는 문화를 지향합니다.

다양성 (Diversity): 다양한 시각과 경험을 존중하며 포용적인 조직 문화를 만들어 갑니다.

1.3 조직 구조

키키테크는 다음과 같은 본부 체계로 운영됩니다:

본부

주요 팀

주요 업무

AI 연구개발본부

ML팀, LLM팀, 데이터팀

AI 모델 연구 및 제품 개발

플랫폼개발본부

백엔드팀, 프론트엔드팀, DevOps팀

서비스 플랫폼 구축 및 운영

사업본부

영업팀, 마케팅팀, 파트너십팀

고객 발굴 및 사업 확장

경영지원본부

인사팀, 재무팀, 법무팀

내부 운영 및 지원




2장. 임직원 행동강령

2.1 기본 윤리 원칙

모든 임직원은 

In [30]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# PDF/Word처럼 긴 문서는 splitting 필요
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
word_chunks = splitter.split_documents(word_contents)

print(len(word_chunks))

13


#### 1-4. ppt 파일 처리

ppt 파일은 엑셀과 유사하게 직접 처리해서 Document 형태로 변환해 줄 필요가 있습니다.

Presentation 라이브러리를 활용해서 데이터를 로드해보겠습니다.

In [ ]:
from pptx import Presentation
from langchain_core.documents import Document

def parse_pptx(file_path: str) -> list[Document]:
    # ppt 파일 로드
    prs = Presentation(file_path)
    docs = []
    # 각 슬라이드를 돌며...
    for slide_num, slide in enumerate(prs.slides, 1):
        texts = []
        # 슬라이드의 모든 요소를 살펴보며 텍스트가 있으면 texts 리스트에 추가.
        for shape in slide.shapes:
            if shape.has_text_frame:
                for para in shape.text_frame.paragraphs:
                    if para.text.strip():
                        texts.append(para.text.strip())
        if texts:
            docs.append(Document(
                page_content="\n".join(texts),
                metadata={
                    "source": file_path,
                    "slide": slide_num,
                }
            ))
    return docs

ppt_contents = parse_pptx("data/키키테크_회사소개.pptx")

print(len(ppt_contents))
print(ppt_contents[0].page_content)

5
(주)키키테크
회사 소개
기술로 연결하는 더 나은 내일
2025
2015
설립연도
350+
임직원 수
480억
2024 매출


각 슬라이드마다 문서가 하나씩 생성된 것을 볼 수 있습니다. ppt 특성상 각 슬라이드에 텍스트가 많은 편이 아니기 때문에 별도로 splitting 과정을 거치진 않겠습니다.

#### 실습

위에서 작성한 코드들을 `parsing.py` 파일에 함수로 구현해 봅시다!

1. excel_parser(file_path)
2. pdf_parser(file_path)
3. ppt_parser(file_path)
4. word_parser(file_path)

저장하고 불러오기

In [ ]:
from langchin_classic.retrievers import EnsembleRetriever

...

### 3. 벡터 유사도를 찾는 함수를 tool로 구현하기

이제 위에서 살펴 본 similarity_search() 메소드를 사용해 입력된 query와 유사한 상위 k개의 문서를 찾는 코드를 작성해 보도록 하겠습니다.

In [43]:
from langchain_core.tools import tool

@tool
def search_documents(query: str) -> str:
    """VectorDB에서 문서를 검색합니다. 키키테크의 제품 정보, 사내 규정, 임직원 및 프로젝트 현황 등의 정보를 확인할 수 있습니다."""
    results = vectorstore.similarity_search(query, k=5) # 상위 5개 문서 추출
    formatted = []
    for doc in results:
        meta = doc.metadata
        # Document 객체는 메타데이터를 포함하고 있어 참고한 문서의 출처를 확인할 수 있습니다.
        source = Path(meta.get("source", "unknown")).name
        file_type = meta.get("file_type", "unknown")
        chunk_id = meta.get("chunk_id", "")
        formatted.append(
            f"[출처: {source} ({file_type}, {chunk_id})]\n{doc.page_content}"
        )
    return "\n\n---\n\n".join(formatted)

print(search_documents.invoke({"query": "키키테크의 주요 수입모델은?"}))

[출처: 키키테크_회사소개.pptx (unknown, )]
(주)키키테크
회사 소개
기술로 연결하는 더 나은 내일
2025
2015
설립연도
350+
임직원 수
480억
2024 매출

---

[출처: 키키테크_AI솔루션_제품카탈로그.pdf (unknown, )]
1. 회사 개요 및 솔루션 포트폴리오
1.1 (주)키키테크 소개
(주)키키테크는 2015년 설립된 국내 대표 AI 솔루션 전문기업입니다. 창립 이래 기업의 디지털
전환을 선도하는 혁신적인 AI 기술을 연구·개발해 왔으며, 현재 금융, 제조, 유통, 의료,
법률 등 다양한 산업군의 350여 개 기업 고객을 보유하고 있습니다.
서울 강남구 테헤란로에 본사를 두고 판교, 부산에 지사를 운영 중이며, AI 연구개발 인력
120명을 포함한 총 350명의 임직원이 고객 가치 창출을 위해 노력하고 있습니다. 2024년
매출액 480억원을 기록하며 3년 연속 30% 이상의 성장세를 이어가고 있습니다.
주요 경쟁력
 LLM 파인튜닝 및 RAG 기술 내재화 — 자체 AI 연구팀 보유
 온프레미스/클라우드/하이브리드 배포 모두 지원
 금융위원회 마이데이터 사업자 허가, ISO 27001 인증 보유
 24시간 365일 SLA 기반 기술 지원 체계

---

[출처: 키키테크_회사소개.pptx (unknown, )]
회사 개요
(주)키키테크는 2015년 설립된 AI 솔루션 전문기업으로, 기업의 디지털 전환을 선도하는 혁신적인 AI 기술을 연구·개발하고 있습니다. 금융, 제조, 유통, 의료, 법률 등 다양한 산업군의 350여 개 기업 고객을 보유하고 있으며, 서울 강남구 본사를 비롯해 판교·부산 지사를 운영 중입니다.
혁
혁신
새로운 기술과
아이디어 탐구
신
신뢰
투명하고 책임감
있는 파트너십
협
협력
다양한 역량의
구성원이 함께
성
성장
개인과 조직이
함께 성장
다
다양성
포용적 조직
문화 지향
비전: 기술로 연결하는 더 나은 내일

---

[출처: 키키테크_회사소개.pptx (unknown, )]
주요 성과 및

### 4. 에이전트에 tool 추가하기

이제 에이전트에 위에서 구현한 tool을 추가해 보도록 하겠습니다!

In [44]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# 모델 이름
model_name = "gpt-5.4-mini"

# LLM 정의
model = init_chat_model(
    model=model_name,
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0.7,
)

agent = create_agent(
    model,
    tools=[search_documents],
    system_prompt="너는 키키테크의 사내 Q&A 챗봇이다. 기업과 관련된 정보에 대해 문서를 꼭 확인하고 답을 생성해야 한다. 문서에서 확인할 수 없는 질문에는 대답하지 말 것."
)

response = agent.invoke(
    {"messages": {"role": "user", "content": "키키테크의 주요 수익모델은 뭐야?"}}
)

print(response['messages'][-1].content)

문서에서 확인되는 범위에서는 **키키테크의 주요 사업/수익원은 기업용 AI 솔루션과 데이터 분석 플랫폼, AI 에이전트 개발 서비스**입니다.

근거:
- 사내 규정 문서: “**기업용 데이터 분석 플랫폼과 AI 에이전트 개발 서비스를 제공합니다**”
- 제품 카탈로그: 키키테크는 **AI 솔루션 전문기업**으로, 금융·제조·유통·의료 등 다양한 산업군의 고객을 대상으로 솔루션을 제공합니다.

다만, **매출 구조(예: 라이선스, 구독, 구축형 프로젝트, 유지보수 비중 등)** 같은 “정확한 수익모델”의 세부 항목은 문서에서 확인되지 않았습니다.
